# VideoLLaMA3 Evaluation on PHLOP Dataset

In [ ]:
%pip install flash-attn --no-build-isolation
%pip install transformers==4.46.3 accelerate==1.0.1
%pip install decord ffmpeg-python imageio opencv-python matplotlib
%pip install -U bitsandbytes
%pip install datasets huggingface_hub

## Configuration

In [ ]:
import os
import sys
import json
import torch
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from huggingface_hub import login

sys.path.insert(0, str(Path(".").resolve()))

from phlop_eval_common import (
    load_phlop_splits, load_json_file, load_qa_from_path,
    get_physical_props, get_taxonomy,
    evaluate_response_quality, build_dynamic_prompt,
    save_and_score_results, EVAL_OPTIONS,
    FINE_TUNE_CONFIGS, get_val_difficulty_filter, filter_qa_by_difficulty,
    build_training_index,
)

REPO_ID = "zimmari-ai/phlop"
HF_TOKEN = os.environ.get("HF_TOKEN", True)
CAMERA_MODE = "static"
NUM_FRAMES = 32
MAX_SAMPLES = 4000

In [ ]:
if isinstance(HF_TOKEN, str) and HF_TOKEN.startswith("hf_"):
    login(token=HF_TOKEN)

## Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "DAMO-NLP-SG/VideoLLaMA3-7B"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    device_map={"": device},
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

## Load Dataset from HuggingFace

In [ ]:
splits = load_phlop_splits(REPO_ID, token=HF_TOKEN)
test_ds = splits["test"]
for name, ds in splits.items():
    print(f"  {name}: {len(ds)} scenes")

## Evaluation

In [ ]:
results = []
fps = 2

for idx in tqdm(range(min(MAX_SAMPLES, len(test_ds))), desc="Evaluating"):
    sample = test_ds[idx]

    video_path = (sample.get("videos") or {}).get(CAMERA_MODE)
    meta_path = (sample.get("metadata") or {}).get(CAMERA_MODE)
    if not video_path or not meta_path:
        continue

    metadata = load_json_file(meta_path)
    physical_props = get_physical_props(metadata)
    taxonomy = get_taxonomy(metadata)
    qa_list = load_qa_from_path((sample.get("qa") or {}).get(CAMERA_MODE))
    if not qa_list:
        continue

    scene_id = sample.get("id", str(idx))

    for option in EVAL_OPTIONS:
        tax = taxonomy if option["is_taxonomy"] else {}
        phys = physical_props if option["is_physics"] else {}

        for qa in qa_list:
            prompt = build_dynamic_prompt(
                taxonomy=tax,
                physical_props=phys,
                question=qa["question"],
                options=qa.get("options"),
                explanation=qa.get("explanation"),
                num_frames=NUM_FRAMES,
                fps=fps,
            )

            conversation = [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": [
                    {"type": "video", "video": {"video_path": video_path, "fps": fps, "max_frames": NUM_FRAMES}},
                    {"type": "text", "text": prompt},
                ]},
            ]
            inputs = processor(conversation=conversation, return_tensors="pt")
            inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
            if "pixel_values" in inputs:
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

            with torch.no_grad():
                output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.1)
            response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

            eval_res = evaluate_response_quality(response, qa["answer"], taxonomy, physical_props)

            results.append({
                "scene_id": scene_id,
                "question": qa["question"],
                "answer": qa["answer"],
                "options": qa.get("options"),
                "response": response,
                "correct": eval_res["correct"],
                "error": eval_res["error"],
                "option": option["name"],
            })

print(f"Collected {len(results)} results")

## Results

In [ ]:
os.makedirs("results", exist_ok=True)
scores = save_and_score_results(results, "results/llama3_results.json")

## Part 2: Fine-tuning

LoRA fine-tuning with 4 difficulty configurations:
1. **easy** - train on easy, validate on rest
2. **easy_medium** - train on easy+medium, validate on rest
3. **hard** - train on hard, validate on rest
4. **full** - full dataset

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset

OUTPUT_DIR = "./llama3_checkpoints"
MAX_STEPS = 50


class PHLOPFineTuneDataset(TorchDataset):
    """Expands scenes into (scene, question) pairs with difficulty filtering."""

    def __init__(self, ds, index, camera_mode="static"):
        self.ds = ds
        self.index = index
        self.camera_mode = camera_mode

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        scene_idx, qa_idx = self.index[idx]
        sample = self.ds[scene_idx]

        video_path = (sample.get("videos") or {}).get(self.camera_mode)
        meta_path = (sample.get("metadata") or {}).get(self.camera_mode)
        metadata = load_json_file(meta_path) if meta_path else {}
        physical_props = get_physical_props(metadata)
        taxonomy = get_taxonomy(metadata)

        qa_list = load_qa_from_path((sample.get("qa") or {}).get(self.camera_mode))
        qa = qa_list[qa_idx] if qa_idx < len(qa_list) else {}

        question = qa.get("question", "")
        answer = qa.get("answer", "")
        if isinstance(answer, list):
            answer = ", ".join(str(a) for a in answer)
        if qa.get("options"):
            question += "\nOptions: " + str(qa["options"])

        prompt = build_dynamic_prompt(
            taxonomy=taxonomy,
            physical_props=physical_props,
            question=question,
            options=qa.get("options"),
            explanation=qa.get("explanation"),
            num_frames=NUM_FRAMES,
        )

        return {
            "video_path": video_path or "",
            "prompt": prompt,
            "answer": str(answer),
            "metadata": metadata,
        }


class LLaMA3DataCollator:
    def __init__(self, processor, fps=2, max_frames=32):
        self.processor = processor
        self.fps = fps
        self.max_frames = max_frames

    def __call__(self, batch):
        conversations = []
        for item in batch:
            conv = [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": [
                    {"type": "video", "video": {"video_path": item["video_path"], "fps": self.fps, "max_frames": self.max_frames}},
                    {"type": "text", "text": item["prompt"]},
                ]},
                {"role": "assistant", "content": item["answer"]},
            ]
            conversations.append(conv)

        texts = [self.processor.apply_chat_template(conv, tokenize=False) for conv in conversations]
        model_inputs = self.processor(text=texts, return_tensors="pt", padding=True, truncation=True)
        model_inputs["labels"] = model_inputs["input_ids"].clone()
        return model_inputs

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
data_collator = LLaMA3DataCollator(processor, fps=2, max_frames=NUM_FRAMES)

saved_checkpoints = []
train_ds = splits.get("train", splits["test"])

for cfg_name, cfg in FINE_TUNE_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Fine-tuning config: {cfg_name}")
    print(f"  Train difficulties: {cfg['train_difficulty']}")
    val_diff = get_val_difficulty_filter(cfg["train_difficulty"]) if cfg["val_on_rest"] else None
    print(f"  Val difficulties: {val_diff}")
    print(f"{'='*60}")

    train_index = build_training_index(train_ds, difficulty_filter=cfg["train_difficulty"], camera_mode=CAMERA_MODE)
    if not train_index:
        print(f"  Skipping {cfg_name}: no training samples.")
        continue

    ft_train = PHLOPFineTuneDataset(train_ds, train_index, camera_mode=CAMERA_MODE)

    ft_model = AutoModelForCausalLM.from_pretrained(
        model_path, trust_remote_code=True,
        device_map={"": device}, torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
    ft_model = get_peft_model(ft_model, lora_config)
    ft_model.print_trainable_parameters()

    ckpt_dir = os.path.join(OUTPUT_DIR, cfg_name)
    training_args = TrainingArguments(
        output_dir=ckpt_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        max_steps=MAX_STEPS,
        bf16=True,
        logging_steps=10,
        save_steps=500,
        remove_unused_columns=False,
        report_to="none",
    )

    trainer = Trainer(
        model=ft_model,
        args=training_args,
        train_dataset=ft_train,
        data_collator=data_collator,
    )
    trainer.train()
    trainer.save_model()
    saved_checkpoints.append((cfg_name, ckpt_dir))
    del ft_model
    torch.cuda.empty_cache()

print(f"\nSaved {len(saved_checkpoints)} checkpoints: {saved_checkpoints}")

## Part 3: Evaluate Fine-tuned Models on Test

In [ ]:
print(f"\n{'Config':<20} {'Accuracy':>10}")
print("-" * 35)

for cfg_name, ckpt_dir in saved_checkpoints:
    from peft import PeftModel

    ft_model = AutoModelForCausalLM.from_pretrained(
        model_path, trust_remote_code=True,
        device_map={"": device}, torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
    ft_model = PeftModel.from_pretrained(ft_model, ckpt_dir)
    ft_model.eval()

    ft_results = []
    for idx in tqdm(range(min(MAX_SAMPLES, len(test_ds))), desc=f"Eval {cfg_name}"):
        sample = test_ds[idx]
        video_path = (sample.get("videos") or {}).get(CAMERA_MODE)
        meta_path = (sample.get("metadata") or {}).get(CAMERA_MODE)
        if not video_path or not meta_path:
            continue

        metadata = load_json_file(meta_path)
        physical_props = get_physical_props(metadata)
        taxonomy = get_taxonomy(metadata)
        qa_list = load_qa_from_path((sample.get("qa") or {}).get(CAMERA_MODE))
        if not qa_list:
            continue

        for qa in qa_list:
            prompt = build_dynamic_prompt(
                taxonomy=taxonomy, physical_props=physical_props,
                question=qa["question"], options=qa.get("options"),
                explanation=qa.get("explanation"), num_frames=NUM_FRAMES,
            )
            conversation = [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": [
                    {"type": "video", "video": {"video_path": video_path, "fps": 2, "max_frames": NUM_FRAMES}},
                    {"type": "text", "text": prompt},
                ]},
            ]
            inputs = processor(conversation=conversation, return_tensors="pt")
            inputs = {k: v.to(ft_model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
            if "pixel_values" in inputs:
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

            with torch.no_grad():
                output_ids = ft_model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.1)
            response = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

            eval_res = evaluate_response_quality(response, qa["answer"], taxonomy, physical_props)
            ft_results.append({"correct": eval_res["correct"], "option": "taxonomy_and_physics"})

    if ft_results:
        acc = sum(r["correct"] for r in ft_results) / len(ft_results)
        print(f"{cfg_name:<20} {acc:>10.4f} ({sum(r['correct'] for r in ft_results)}/{len(ft_results)})")

    os.makedirs("results", exist_ok=True)
    save_and_score_results(ft_results, f"results/llama3_ft_{cfg_name}.json")

    del ft_model
    torch.cuda.empty_cache()